# 리포트 78 — 앙각 7 점을 10 m 한 자리에서 재고, 47 행 중 46 행만 판정에 쓴다

> ### 한 일
> **관측 앙각 7 점을 10 m 구면 한 자리에 고정해 세 팔로 같은 표적을 재고, 완결된 46 행만 인용 대상으로 갈랐다.**

### 결과
1. 잰 자리는 하나다 — 반경 10 m [^1] 구면, 방위 0° [^2], 앙각 +0° [^3] 에서 -90° [^4] 까지 7 점.
2. 한 앙각마다 자세 4,096 개 [^5] 를 PRF 19,700 Hz [^6] 로 태웠고, 표적은 matrice4e [^7] 하나다.
3. 그 자리는 원거리장 경계의 안쪽이라 근접장 판이고, 우리 커널의 조명 규약은 «spherical wave at 10 m [^8]» 다.
4. 원장 47 행 중 46 행이 `n_missing = 0` 이고, 물리 스위치 팔의 −15°(0 개 [^9] 빠짐) 와 −45°(0 개 [^10] 빠짐) 두 행은 판정에서 뺀다.
5. 경로 수는 팔 사이에서 예산 축이고 한 팔 안에서 기하 축이다 — 같은 7 점에서 규칙값 팔은 6 [^11]~13 [^12] 개, `--spp` 로 광선을 22.5 배 올린 팔은 127 [^13]~352 [^14] 개를 센다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기하 | 반경 10 m 구면 위에서 방위 하나·앙각 일곱. 송신과 수신은 같은 자리다(baseline 0) — `benchmark/elevation_sweep_md.py:83,175` |
| 표적·회전 | matrice4e 메쉬 전체(동체·팔·로터·짐벌)에 첫 충돌 가림. 로터 넷은 덱과 같은 결정론 RPM 이라 바뀌는 축은 앙각 하나다 |
| 조명 | 우리 커널은 구면파로 조명하고, PathSolver 는 송수신 위치를 실제 기하로 놓는다. 광선 수의 기본값을 거리만 보는 규칙 `(R/3)²×1M` 이 정하고, p250M·p1G·p4G 팔은 `--spp` 로 그 값을 덮어쓴다 |
| 분석 대역 | 추적 대역은 앙각마다 그 앙각의 날개끝 주파수로 다시 잡고, 고정 대역은 덱의 −15° 대역을 그대로 쓴다 — 두 정의는 «대역은 두 가지로 잰다» 절에 원장 문장 그대로 싣는다 |
| 완결성 | `rows[i].n_missing` 은 시계열에 0 으로 남은 자세 수다. 이 조각의 표는 `(engine, el_deg, n_missing == 0)` 으로 행을 찾아 만들었다 |

### 재현

```bash
SIONNA2_GPU=3 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --engine ours --shard 0 --nshards 8
SIONNA2_GPU=3 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --engine sionna --shard 0 --nshards 8
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/elevation_sweep_md.py --merge
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_fig_el_geometry.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/render_el15_scene.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/build_el15_scenario_fig.py
```

| | |
|---|---|
| 출력 | `outputs/elevation_sweep_md.json`, `outputs/elevation_sweep_md.npz`, `outputs/figures/ch1_f0_geometry.png`, `outputs/figures/el15_scenario.png` |
| 소요 | 한 행마다 GPU 누적 14 분 ~ 2 시간 21 분 · 샤드 8 개 병렬. 병합과 그림은 CPU 로 수 초 |
| 비고 | `--merge` 는 샤드 폴더 `outputs/elev_sweep_shards/` 를 읽어 원장 두 개를 다시 쓴다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 40 «자세와 가림»](40_md-attitude.ipynb) | 지상 레이더가 기체를 아래에서 본다는 **기체 자세** 축 — 이 권이 바꾸는 앙각은 **수신 기하**이고 그 자세와 다른 축이다 |
| [편 36 «두 엔진»](36_md-two-engines.ipynb) | 두 엔진이 날개끝 주파수 아래에서 겹치고 그 위에서 갈린다는 −15° 한 점의 결과 |

---

## 잰 자리는 한 자리다

앙각 하나만 바꾼다. 거리·방위·로터 RPM·자세 격자는 같은 값으로 얼려 뒀으므로, 팔 사이와 앙각 사이에서 달라진 것은 시선 방향 하나다.

로터 설정을 원장은 이렇게 적는다 — «덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 [^15]».

| 설정 | 값 |
|---|---|
| 표적 | matrice4e [^7] |
| 반송파 | 3.50e+09 Hz [^16] |
| 거리 — 구면 반경 | 10 m [^1] |
| 방위 | 0° [^2] |
| 앙각 | +0° [^3] 에서 -90° [^4] 까지 7 점 |
| 자세 | 4,096 개 [^5] |
| PRF | 19,700 Hz [^6] |
| 플래시 박자 — 예측 입력 | 126.67 Hz [^17] |
| 날개끝 주파수 — 앙각 0° | 1272.9 Hz [^18] |

## 그 자리는 어느 원거리장 정의를 쓰느냐로 갈린다

> ⭐사용자 지시로 10 m 고정. ⚠원거리장 경계 2D²/λ ≈ 14.08 m 의 **안쪽**이라 근거리장 판이다 — 우리 커널은 range_m 구면파로 처리하고 PathSolver 는 실제 기하라 둘 다 다룰 수 있지만, 평면파 원거리장 값과 직접 비교하면 안 된다. [^19]

matrice4e·3.5 GHz 에서 2D²/λ 는 **D 를 무엇으로 잡느냐에 따라 두 값**이다 — 수평 최대치수 0.59 m [^20] 로 잡으면 8.26 m [^21], 메쉬 3D 대각 0.78 m [^22] 로 잡으면 14.08 m [^23] 다. 10 m 는 그 **사이**에 있다. 곧 이 판은 보수적 정의(3D 대각)에서 경계 안쪽이고, 수평 최대치수 정의에서는 경계 밖이다. 이 권은 «경계» 를 쓸 때 **정의를 값과 함께** 적는다.

그것이 이 판에서 실제로 바꾸는 것은 **둘**이다.

1. 우리 커널은 표적을 구면파로 조명한다 — 원장의 조명 규약은 «spherical wave at 10 m [^8]» 다.
2. 평면파 원거리장에서 잰 레벨과 이 판의 레벨을 같은 줄에 놓으려면 **환산 몫을 적는다.** 그 몫의 크기는 이미 재어 뒀다 — 같은 기체·같은 반송파·같은 얼린 격자·앙각 −15° 한 점에서 구면파와 평면파의 차이는 8 m 에서 레벨 0.52 dB [^24] · 맵 코사인 0.998 [^25], 경계 밖 15 m 에서 0.29 dB [^26] · 0.997 [^27] 다. 차이는 거리와 함께 매끄럽게 줄고 **경계를 그대로 지나간다.**

다음 셋은 경계와 무관한 **판 조건**이다 — 어느 거리에서도 같다.

- PathSolver 는 송수신 위치를 실제 기하로 놓는다. 그 계산이 받는 것은 두 자리의 좌표뿐이다.
- 광선 수 11,111,111 개 [^28] 는 `(R/3)²×1M` 이라는 **거리만 보는** 규칙이 정한다.
- 표면 격자는 «얼린 격자(자세 합집합 bbox), λ/12 [^29]» 다 — 자세마다 다시 잡지 않으므로 앙각 사이의 차이는 격자가 아니라 자세와 시선에서 온다.

![experiment scenario rendered with Sionna RT](../outputs/figures/el15_scenario.png)

**그림 1.** 이 판은 표적을 어느 자리에서 보고, 그 자리에서 무엇이 보이나?

위 칸은 기하다 — 15 m 구면 위 앙각 일곱 점과, 그 안쪽을 지나는 원거리장 경계선. 아래 칸은 그 일곱 자리에서 **레이더가 실제로 보는 표적**이고 `benchmark/render_el15_scene.py` 가 Sionna RT 로 낸 렌더다. 왼쪽 끝(앙각 0°)은 로터를 옆에서 봐 블레이드가 선으로 보이고, 오른쪽 끝(−90°)은 로터 원반이 열리는 대신 **동체가 가운데를 덮는다** — 아래 절이 가르는 두 몫이 이것이다.

## 메쉬를 통째로 넣은 대가는 두 몫이 겹쳐 있다

메쉬는 통째로 넣었다 — 동체·팔·로터·짐벌이 다 들어 있고 첫 충돌 가림이 켜져 있다. 그래서 앙각을 내리면 두 가지가 함께 움직인다.

1. 날개끝 속도의 시선 방향 성분이 cos(el) 로 준다.
2. 동체가 로터와 센서 사이로 들어온다 — 나딧에서는 동체 원반이 로터를 덮는다.

아래 표의 «날개끝 주파수» 는 1 번만 담는 **입력값**이다. `f_tip_at()`(`benchmark/elevation_sweep_md.py:205`) 이 로터 지름·회전수에서 f_tip = 2·(2π f_rev R)/λ · cos(el) 로 내며, 앙각 0° 의 1272.9 Hz [^18] 에서 −90° 의 0.0 Hz [^30] 로 간다. 그것은 cos(el) 열에 앙각 0° 값을 곱한 수와 같고, 원장에서 앙각 0° 를 가진 여섯 팔이 전부 같은 값을 싣는다 — 이 열은 앙각 하나의 함수다. 이 표에서 이 판이 잰 열은 `빠진 자세` 하나이고, 일곱 행 모두 0 이라 우리 커널의 일곱 점을 그대로 인용한다.

| 앙각 [°] | cos(el) | 날개끝 주파수 [Hz] | 빠진 자세 |
|---|---|---|---|
| +0 | 1.0000 | 1272.9 | 0 |
| -15 | 0.9659 | 1229.5 | 0 |
| -30 | 0.8660 | 1102.4 | 0 |
| -45 | 0.7071 | 900.1 | 0 |
| -60 | 0.5000 | 636.5 | 0 |
| -75 | 0.2588 | 329.5 | 0 |
| -90 | 0.0000 | 0.0 | 0 |

출처 [^31]

2 번 몫은 대조군이 가른다. 동체의 «면만» 빼고 정점을 남겨 bbox 와 광선 격자를 보존하는 `ours_free`(`benchmark/elevation_sweep_md.py:117-122, 132-133`) 가 그것이고, 축은 «동체가 막느냐» 하나다. 그 팔은 스크립트에 배선돼 있고 원장에도 샤드에도 행이 없어 «다음 단계» 첫 줄이다.

지금 배선은 `keep = np.asarray(fp.g) == "prop"` 이라 prop 아닌 면을 **전부** 뺀다 — `DRONE_GROUP_MAT` 기준으로 body·canopy·arm·motor·gear·camera·accent·battery·pcb 가 함께 빠지므로 가림과 정적 산란체(DC 분모)가 한 축에 묶인다. 가림만 가르려면 `keep` 을 «body·canopy 만 뺀다» 로 좁혀 돌린다.

## 세 팔이 같은 자리에서 낸 것

![micro-Doppler maps versus elevation](../outputs/figures/ch1_f1_maps.png)

**그림 2.** 세 팔은 같은 자리에서 앙각을 내릴 때 무엇을 냈나?

위 줄이 우리 커널(SBR + PO), 가운데가 PathSolver, 아래가 광선 예산을 올린 PathSolver(`sionna_p250000000`) 다. 판마다 자기 최댓값으로 정규화했으므로 판 사이 레벨 비교는 이 그림 밖이다. 그림은 7 점 중 네 점을 싣고, 일곱 점 전부의 완결성은 아래 표에 있다.

## 대역은 두 가지로 잰다

- 추적 대역 — ⭐정본 — 앙각마다 그 앙각의 f_tip 으로 0.35~1.0 배 [^32]
- 고정 대역 — 덱의 −15° 대역(430~1229 Hz) 고정 — 앙각이 내려가면 비어 간다 [^33]

둘을 함께 내는 이유는 «고정 대역을 쓰면 어디서 무너지나» 가 그 자체로 결과이기 때문이다. 그 판정은 [편 79 «대역 추적»](79_el-band-tracking.ipynb) 에 있다.

## 어느 행을 인용해도 되나 — 47 행 중 46 행

원장은 47 행이고 그중 46 행이 `n_missing = 0` 이다. 나머지 두 행은 물리 스위치 팔의 −15° 와 −45° 이고 빠진 자세가 각각 0 개 [^9] · 0 개 [^10] 다 — 그 자리에 0 이 박혀 있어 스펙트럼과 레벨이 그만큼 눌린다. 이 권은 그 두 행을 판정에서 뺀다.

아래 표가 세 팔 밖의 행 전부다. 예산 사다리(광선 1 G · 4 G)와 물리 스위치 팔이 여기 산다.

| 팔 | 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|---|
| sionna_p1000000000 | +0 | 0 | 471 |
| sionna_p1000000000 | -15 | 0 | 736 |
| sionna_p1000000000 | -30 | 0 | 809 |
| sionna_p1000000000 | -45 | 0 | 1065 |
| sionna_p1000000000 | -60 | 0 | 1143 |
| sionna_p1000000000 | -75 | 0 | 1241 |
| sionna_p1000000000 | -90 | 0 | 1370 |
| sionna_p250000000_phys | +0 | 0 | 33 |
| sionna_p250000000_phys | -15 | 0 | 48 |
| sionna_p250000000_phys | -30 | 0 | 52 |
| sionna_p250000000_phys | -45 | 0 | 64 |
| sionna_p250000000_phys | -60 | 0 | 66 |
| sionna_p250000000_phys | -75 | 0 | 57 |
| sionna_p250000000_phys | -90 | 0 | 64 |
| sionna_p4000000000 | +0 | 0 | 2008 |
| sionna_p4000000000 | -15 | 0 | 2767 |
| sionna_p4000000000 | -45 | 0 | 4136 |
| sionna_p4000000000 | -60 | 0 | 4707 |
| sionna_p4000000000 | -75 | 2,048 | 5197 |
| sionna_phys | +0 | 0 | 5 |
| sionna_phys | -15 | 0 | 2 |
| sionna_phys | -30 | 0 | 7 |
| sionna_phys | -45 | 0 | 8 |
| sionna_phys | -60 | 0 | 4 |
| sionna_phys | -75 | 0 | 3 |
| sionna_phys | -90 | 0 | 6 |

출처 [^31]

이 권이 이 원장을 읽는 규칙은 셋이다.

1. 판정은 대역 몫으로 하고 `level_db` 는 [편 87 «광선 예산»](87_budget-not-physics.ipynb) 이 같은 엔진 안에서 다룬다 — 팔마다 정규화가 다르다.
2. −90° 에서 추적 대역의 폭이 0 이라 그 칸이 `null` 이다. `null` 은 0 이 아니라 «잴 수 없다» 는 표시이고, [편 79 «대역 추적»](79_el-band-tracking.ipynb) 이 «측정 불가» 로 적는다.
3. 행 번호는 병합할 때마다 밀린다. 이 조각의 표는 `(engine, el_deg, n_missing == 0)` 으로 행을 찾아 만들었다.

## 경로 수는 팔 사이에서만 예산 축이다

10 m 한 자리에서 규칙 `(R/3)²×1M` 은 11,111,111 개 [^28] 로 고정이고, 일곱 점의 예산이 모두 같은 값 하나다. 그래서 규칙값 팔의 6 [^11]~13 [^12] 개와 250M 팔의 127 [^13]~352 [^14] 개를 가른 것은 `--spp` 로 규칙값의 22.5 배를 쏜 설정 하나다(`benchmark/elevation_sweep_md.py:88,150`).

예산이 고정된 한 팔 **안에서** 경로 수가 앙각을 따라 움직이는 몫은 시선 기하가 정한다. 250M 팔은 앙각 0° 에서 −90° 로 가는 동안 단조로 늘고, 규칙값 팔은 6~13 사이에서 흔들려 그 추이를 내지 않는다 — 아래 두 표가 같은 열을 앙각별로 싣는다.

⇒ 팔 사이에서 경로 수가 다르면 예산 설정을 먼저 보고, 한 팔 안에서 달라지면 시선 기하를 본다. 인용한 네 수는 자세 4,096 개 중앙값의 최소·최대이고, 자세 하나하나의 경로 수는 그보다 넓게 흩어진다. 경로 수를 산란 세기로 읽는 해석은 [편 02 «경로가 무엇을 세나»](02_engine-paths.ipynb) 와 [편 83 «물리 스위치»](83_physics-single-axis.ipynb) 가 다룬다.

**PathSolver(`sionna`)**

| 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|
| +0 | 0 | 9 |
| -15 | 0 | 7 |
| -30 | 0 | 6 |
| -45 | 0 | 12 |
| -60 | 0 | 13 |
| -75 | 0 | 13 |
| -90 | 0 | 12 |

출처 [^31]

**PathSolver(`sionna_p250000000`) — 광선 예산을 올린 팔**

| 앙각 [°] | 빠진 자세 | 경로 수 중앙값 |
|---|---|---|
| +0 | 0 | 127 |
| -15 | 0 | 159 |
| -30 | 0 | 211 |
| -45 | 0 | 268 |
| -60 | 0 | 287 |
| -75 | 0 | 323 |
| -90 | 0 | 352 |

출처 [^31]

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 가림만 끄고 산란체는 남기는 팔을 배선해 같은 일곱 점에서 돌린다 — 지금의 `ours_free` 는 프로펠러만 남기는 팔이라 분모까지 바뀐다 | 가림이 대역 몫을 얼마나 지우는지가 정적 성분 변화와 분리돼 나온다 | `benchmark/elevation_sweep_md.py` 가림 축 · **새 계산이 필요하다** |
| −60° 와 −75° 사이를 다섯 점 더 잰다 | 고정 대역이 대역외 바닥에 닿는 앙각이 15° 격자 안에서 특정된다 | `benchmark/elevation_sweep_md.py --els` · [편 79 «대역 추적»](79_el-band-tracking.ipynb) |
| 디스크에 8/8 로 차 있는 물리 팔 샤드를 병합한다 | 완결 행이 46 행에서 늘고 물리 팔이 두 점 아닌 다섯 점 이상에서 선다 | `benchmark/elevation_sweep_md.py --merge` (CPU) — 기존 원장을 다시 쓴다 |
| 평면파 조명으로 −90° 한 판을 더 돌린다 | 나딧 잔여에서 근접장 몫과 격자 몫이 직접 갈린다 | `benchmark/elevation_sweep_md.py` 조명 축 · [편 82 «나딧 잔여»](82_el-nadir-floor.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 33개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/elevation_sweep_md.json` | `_meta.range_m` | 10 |
| [^2] | `outputs/report07_three_engines.json` | `_meta.az_deg` | 0 |
| [^3] | `outputs/elevation_sweep_md.json` | `_meta.elevations_deg[0]` | 0 |
| [^4] | `outputs/elevation_sweep_md.json` | `_meta.elevations_deg[6]` | -90 |
| [^5] | `outputs/elevation_sweep_md.json` | `rows[0].n_poses` | 4096 |
| [^6] | `outputs/elevation_sweep_md.json` | `_meta.prf_hz` | 19700 |
| [^7] | `outputs/elevation_sweep_md.json` | `_meta.drone` | matrice4e |
| [^8] | `outputs/elevation_sweep_md.json` | `_meta.ours_illumination` | spherical wave at 10 m |
| [^9] | `outputs/elevation_sweep_md.json` | `rows[41].n_missing` | 0 |
| [^10] | `outputs/elevation_sweep_md.json` | `rows[43].n_missing` | 0 |
| [^11] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[0]` | 6 |
| [^12] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_npaths_min_max[1]` | 13 |
| [^13] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[0]` | 127 |
| [^14] | `outputs/ch1_elevation_figdata.json` | `gates.G5_sionna_p250000000_npaths_min_max[1]` | 352 |
| [^15] | `outputs/elevation_sweep_md.json` | `_meta.rotor_ko` | 덱과 같은 결정론 패턴(OU 프리셋 아님) — 축을 하나만 바꾼다 |
| [^16] | `outputs/elevation_sweep_md.json` | `_meta.fc_hz` | 3.5e+09 |
| [^17] | `outputs/elevation_sweep_md.json` | `_meta.f_flash_hz` | 126.7 |
| [^18] | `outputs/elevation_sweep_md.json` | `rows[0].f_tip_hz` | 1273 |
| [^19] | `outputs/elevation_sweep_md.json` | `_meta.range_why_ko` | ⭐사용자 지시로 10 m 고정. ⚠원거리장 경계 2D²/λ ≈ 14.08 m 의 **안쪽**이라 근… |
| [^20] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.D_horizontal_m` | 0.5947 |
| [^21] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.farfield_m` | 8.259 |
| [^22] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.D_diag3d_m` | 0.7764 |
| [^23] | `outputs/report15_probe.json` | `airframes.matrice4e.physics.farfield_diag3d_m` | 14.08 |
| [^24] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.8.level_diff_db` | 0.5152 |
| [^25] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.8.map_cosine` | 0.9976 |
| [^26] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.15.level_diff_db` | 0.2936 |
| [^27] | `outputs/nearfield_sphere_vs_plane.json` | `ranges.15.map_cosine` | 0.9974 |
| [^28] | `outputs/elevation_sweep_md.json` | `_meta.sionna_spp` | 11111111 |
| [^29] | `outputs/elevation_sweep_md.json` | `_meta.grid_ko` | 얼린 격자(자세 합집합 bbox), λ/12 |
| [^30] | `outputs/elevation_sweep_md.json` | `rows[6].f_tip_hz` | 0 |
| [^31] | `outputs/elevation_sweep_md.json` | `rows` | (47행 표) |
| [^32] | `outputs/elevation_sweep_md.json` | `_meta.band_track_ko` | ⭐정본 — 앙각마다 그 앙각의 f_tip 으로 0.35~1.0 배 |
| [^33] | `outputs/elevation_sweep_md.json` | `_meta.band_fixed_ko` | 덱의 −15° 대역(430~1229 Hz) 고정 — 앙각이 내려가면 비어 간다 |